# Notebook 3: Quantum Machine Learning — Classification

This notebook benchmarks five QML classification methods on synthetic datasets:

| Model | Approach |
|-------|----------|
| **VQC** | Variational Quantum Classifier — parameterized encoding + ansatz |
| **QSVM** | Quantum Support Vector Machine — quantum kernel + classical SVM |
| **QCNN** | Quantum Convolutional Neural Network — conv + pooling layers |
| **Quantum Reservoir** | Fixed random circuit + classical readout |
| **Data Re-uploading** | Repeated data encoding + rotation layers |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

## Dataset: Moons

A classic non-linearly separable 2D dataset.

In [ ]:
np.random.seed(42)
X, y = make_moons(n_samples=100, noise=0.15, random_state=42)

# Scale to [0, pi] range for quantum encoding
scaler = MinMaxScaler(feature_range=(0, np.pi))
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

plt.figure(figsize=(6, 4))
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c='blue', label='Class 0', alpha=0.7)
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c='red', label='Class 1', alpha=0.7)
plt.title('Moons Dataset (training set)')
plt.legend()
plt.show()

print(f"Train: {len(X_train)} samples, Test: {len(X_test)} samples")

---
## 1. VQC — Variational Quantum Classifier

In [ ]:
from qforge.algo.vqc import VQC
from qforge.algo.optimizers import Adam

vqc = VQC(n_qubits=2, n_layers=3, n_classes=2)

vqc_params, vqc_history = vqc.fit(
    X_train, y_train,
    optimizer=Adam(lr=0.05),
    steps=60,
    batch_size=20
)

print(f"VQC train accuracy: {vqc.score(X_train, y_train):.2%}")
print(f"VQC test accuracy:  {vqc.score(X_test, y_test):.2%}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(vqc_history, linewidth=2)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('VQC Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

---
## 2. QSVM — Quantum Support Vector Machine

In [ ]:
from qforge.algo.qsvm import QSVM

qsvm = QSVM(n_qubits=2, n_layers=2, feature_map='zz', C=1.0)
qsvm.fit(X_train, y_train)

print(f"QSVM train accuracy: {qsvm.score(X_train, y_train):.2%}")
print(f"QSVM test accuracy:  {qsvm.score(X_test, y_test):.2%}")

In [ ]:
# Visualize the quantum kernel matrix
K = qsvm.kernel_matrix(X_train)

plt.figure(figsize=(6, 5))
plt.imshow(K, cmap='viridis')
plt.colorbar(label='Kernel value')
plt.title('QSVM Kernel Matrix (training set)')
plt.xlabel('Sample index')
plt.ylabel('Sample index')
plt.show()

---
## 3. QCNN — Quantum Convolutional Neural Network

In [ ]:
from qforge.algo.qcnn import QCNN

# QCNN requires n_qubits to be a power of 2
# Pad features to 4 dimensions
X_train_pad = np.hstack([X_train, np.zeros((len(X_train), 2))])
X_test_pad = np.hstack([X_test, np.zeros((len(X_test), 2))])

qcnn = QCNN(n_qubits=4, n_classes=2)

qcnn_params, qcnn_history = qcnn.fit(
    X_train_pad, y_train,
    optimizer=Adam(lr=0.05),
    steps=60,
    batch_size=20
)

print(f"QCNN train accuracy: {qcnn.score(X_train_pad, y_train):.2%}")
print(f"QCNN test accuracy:  {qcnn.score(X_test_pad, y_test):.2%}")

---
## 4. Quantum Reservoir Computing

In [ ]:
from qforge.algo.reservoir import QuantumReservoir

qr = QuantumReservoir(
    n_qubits=4,
    n_layers=3,
    n_readout=3,        # measure X, Y, Z expectations
    task='classification',
    random_state=42
)

qr.fit(X_train, y_train)

print(f"Reservoir train accuracy: {qr.score(X_train, y_train):.2%}")
print(f"Reservoir test accuracy:  {qr.score(X_test, y_test):.2%}")

---
## 5. Data Re-uploading Classifier

In [ ]:
from qforge.algo.data_reuploading import DataReuploadingClassifier

drc = DataReuploadingClassifier(
    n_qubits=1,   # single-qubit classifier!
    n_layers=6,
    n_classes=2
)

drc_params, drc_history = drc.fit(
    X_train, y_train,
    optimizer=Adam(lr=0.05),
    steps=80,
    batch_size=20
)

print(f"DataReuploading train accuracy: {drc.score(X_train, y_train):.2%}")
print(f"DataReuploading test accuracy:  {drc.score(X_test, y_test):.2%}")

---
## Comparison

In [ ]:
results = {
    'VQC': vqc.score(X_test, y_test),
    'QSVM': qsvm.score(X_test, y_test),
    'QCNN': qcnn.score(X_test_pad, y_test),
    'Reservoir': qr.score(X_test, y_test),
    'DataReupl.': drc.score(X_test, y_test),
}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(results.keys(), results.values(), color=['#4C72B0', '#55A868', '#C44E52', '#8172B3', '#CCB974'])
ax.set_ylabel('Test Accuracy')
ax.set_title('QML Classification — Model Comparison (Moons dataset)')
ax.set_ylim(0, 1.05)
for bar, acc in zip(bars, results.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f'{acc:.0%}',
            ha='center', fontsize=11)
plt.tight_layout()
plt.show()

---
## Decision Boundaries

Visualize how each model partitions the feature space.

In [ ]:
def plot_decision_boundary(model, X, y, title, pad=False, ax=None):
    h = 0.1
    x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]
    if pad:
        grid = np.hstack([grid, np.zeros((len(grid), 2))])
    Z = model.predict(grid).reshape(xx.shape)

    if ax is None:
        ax = plt.gca()
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='blue', s=15, alpha=0.7)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='red', s=15, alpha=0.7)
    ax.set_title(title)

fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))

plot_decision_boundary(vqc, X_test, y_test, 'VQC', ax=axes[0])
plot_decision_boundary(qsvm, X_test, y_test, 'QSVM', ax=axes[1])
plot_decision_boundary(qcnn, X_test, y_test, 'QCNN', pad=True, ax=axes[2])
plot_decision_boundary(qr, X_test, y_test, 'Reservoir', ax=axes[3])
plot_decision_boundary(drc, X_test, y_test, 'Data Reuploading', ax=axes[4])

plt.suptitle('Decision Boundaries', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()